# Rome Travel Guide AI

An intelligent travel assistant powered by **RAG (Retrieval Augmented Generation)** that answers questions about visiting Rome.

**Concepts demonstrated:**
- Document loading and text chunking with LangChain
- OpenAI embeddings and ChromaDB vector store
- Similarity search and retrieval
- RetrievalQA chain for grounded answers
- Gradio UI for interactive Q&A

## 1. Setup and Imports

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API key from .env
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client ready.")

OpenAI client ready.


In [3]:
# LangChain components
from langchain_openai import OpenAIEmbeddings, OpenAI as LangChainOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_classic.chains import RetrievalQAWithSourcesChain

print("LangChain components imported successfully.")

LangChain components imported successfully.


## 2. Load the Travel Guide Data

We load a text file containing information about Rome scraped from a travel guide website.
The data covers attractions, food, transport, neighborhoods, day trips, and practical tips.

In [4]:
DATA_FILE = "rome_travel_guide_data.txt"

loader = TextLoader(DATA_FILE, encoding="utf-8")
raw_documents = loader.load()

print(f"Loaded {len(raw_documents)} document(s)")
print(f"Total characters: {len(raw_documents[0].page_content):,}")
print(f"\nFirst 300 characters:\n{raw_documents[0].page_content[:300]}...")

Loaded 1 document(s)
Total characters: 19,137

First 300 characters:
Source: https://www.rome-travel-guide.com/attractions/colosseum
Title: The Colosseum - Rome's Iconic Amphitheatre
Content:
The Colosseum, also known as the Flavian Amphitheatre, is Rome's most iconic landmark and one of the New Seven Wonders of the World. Built between 70-80 AD under emperors Vespas...


## 3. Split Documents into Chunks

Large documents need to be split into smaller pieces for effective retrieval.
We use `RecursiveCharacterTextSplitter` which tries to split on natural boundaries
(paragraphs, sentences, spaces) to keep chunks semantically coherent.

**Key parameters:**
- `chunk_size=1000` - each chunk is ~1000 characters
- `chunk_overlap=150` - chunks overlap by 150 characters to preserve context across boundaries

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

documents = text_splitter.split_documents(raw_documents)

print(f"Split into {len(documents)} chunks")
print(f"\n--- Example Chunk (Chunk 3) ---")
print(documents[3].page_content)
print(f"\n--- Metadata ---")
print(documents[3].metadata)

Split into 57 chunks

--- Example Chunk (Chunk 3) ---
---END OF SOURCE---

--- Metadata ---
{'source': 'rome_travel_guide_data.txt'}


## 4. Create Embeddings and Vector Store

**Embeddings** convert text into numerical vectors where similar meanings produce similar vectors.
We store these vectors in **ChromaDB**, an open-source vector database, for fast similarity search.

In [6]:
# Initialize OpenAI embeddings model
embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

# Create ChromaDB vector store from our document chunks
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="rome_travel_guide"
)

print(f"Vector store created with {vector_store._collection.count()} embedded chunks")

# Inspect one stored embedding
stored = vector_store._collection.get(include=["embeddings", "documents"], limit=1)
print(f"\nEmbedding dimensions: {len(stored['embeddings'][0])}")
print(f"Sample text: {stored['documents'][0][:100]}...")

Vector store created with 57 embedded chunks

Embedding dimensions: 1536
Sample text: Source: https://www.rome-travel-guide.com/attractions/colosseum
Title: The Colosseum - Rome's Iconic...


## 5. Test Similarity Search

Before building the full RAG chain, let's verify the vector store retrieves relevant chunks.
We run a similarity search query and check if the returned documents match our question.

In [7]:
test_query = "What are the best pasta dishes to try in Rome?"
print(f"Query: '{test_query}'\n")

similar_docs = vector_store.similarity_search(test_query, k=2)

for i, doc in enumerate(similar_docs):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300])
    print(f"Source: {doc.metadata.get('source', 'N/A')}\n")

Query: 'What are the best pasta dishes to try in Rome?'

--- Result 1 ---
Content:
Roman cuisine is characterized by simple, flavorful dishes made with high-quality local ingredients. The four classic Roman pasta dishes are: Cacio e Pepe (pecorino cheese and black pepper), Carbonara (guanciale, egg, pecorino, and black pepper), Amatriciana (guanciale, tomato, and pecorino
Source: rome_travel_guide_data.txt

--- Result 2 ---
Trastevere is one of the most popular neighborhoods for dining, known for its charming cobblestone streets and lively atmosphere. Traditional trattorias like Da Enzo al 29 and Tonnarello serve authentic Roman dishes at reasonable prices, though expect long waits during peak hours. Testaccio is consi
Source: rome_travel_guide_data.txt



In [8]:
# Test with a different query to verify relevance
test_query_2 = "How do I get from the airport to the city center?"
print(f"Query: '{test_query_2}'\n")

similar_docs_2 = vector_store.similarity_search(test_query_2, k=2)

for i, doc in enumerate(similar_docs_2):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:300])
    print()

Query: 'How do I get from the airport to the city center?'

--- Result 1 ---
Content:
Rome is served by two airports. Leonardo da Vinci Airport (Fiumicino, FCO) is the main international airport, located 35 km southwest of the city center. The Leonardo Express train connects Fiumicino to Roma Termini station in 32 minutes for 14 euros. Trains depart every 15 minutes from 6:2

--- Result 2 ---
fare is 3 euros, with a base rate of 1.10 euros per kilometer. Ride-hailing apps like Uber operate in Rome but primarily with licensed taxi drivers. Walking is often the best way to explore the historic center, as many major attractions are within walking distance of each other.



## 6. Build the RAG Chain

Now we assemble the full RAG pipeline using LangChain's `RetrievalQAWithSourcesChain`:
1. **Retriever** - fetches the top-k most relevant chunks from the vector store
2. **LLM** - generates an answer based on the retrieved context
3. **Chain** - connects retriever and LLM, passing sources along with the answer

In [9]:
# Configure the retriever (top 3 most relevant chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Configure the LLM (low temperature for factual answers)
llm = LangChainOpenAI(
    temperature=0,
    openai_api_key=openai_api_key
)

# Build the RAG chain
qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG chain ready!")

RAG chain ready!


In [10]:
# Test the chain
test_question = "What should I wear when visiting churches in Rome?"
print(f"Question: {test_question}\n")

result = qa_chain.invoke({"question": test_question})

print("--- Answer ---")
print(result.get("answer", "No answer"))
print(f"\n--- Sources ---")
print(result.get("sources", "No sources"))

Question: What should I wear when visiting churches in Rome?

--- Answer ---
 When visiting churches in Rome, it is recommended to dress modestly by covering your shoulders and knees. It is also advised to carry a light scarf or shawl in case of hot weather. It is important to be aware of pickpockets and to use anti-theft bags to keep valuables secure. It is also recommended to validate bus or tram tickets to avoid fines. It is important to be cautious of common scams and to only use licensed taxis. It is also advised to check menu prices before ordering at restaurants and to avoid buying from unauthorized street vendors. In case of emergency, dial 112, 113, or 118. 


--- Sources ---
https://www.rome-travel-guide.com/practical/tips-and-etiquette, rome_travel_guide_data.txt


In [11]:
# Test with more questions
questions = [
    "How much does a metro ticket cost?",
    "What is Ostia Antica and how do I get there?",
    "Where is the best neighborhood for dinner?"
]

for q in questions:
    print(f"Q: {q}")
    r = qa_chain.invoke({"question": q})
    print(f"A: {r.get('answer', 'N/A').strip()}")
    print(f"Sources: {r.get('sources', 'N/A')}")
    print("-" * 60)

Q: How much does a metro ticket cost?
A: A single BIT ticket costs 1.50 euros.
Sources: rome_travel_guide_data.txt
------------------------------------------------------------
Q: What is Ostia Antica and how do I get there?
A: Ostia Antica is an ancient Roman port city located 30 km southwest of Rome. It can be reached by train from Roma Porta San Paolo station and is open Tuesday to Sunday. Entrance costs 12 euros and visitors should allow 2-3 hours for a visit. It is recommended to bring water and sun protection.
Sources: https://www.rome-travel-guide.com/daytrips/ostia-antica, rome_travel_guide_data.txt
------------------------------------------------------------
Q: Where is the best neighborhood for dinner?
A: The best neighborhoods for dinner are Trastevere, Testaccio, the Jewish Ghetto, and Pigneto.
Sources: https://www.rome-travel-guide.com/food/best-neighborhoods-for-food, rome_travel_guide_data.txt
------------------------------------------------------------


## 7. Gradio Interface

Let's wrap the RAG chain in a user-friendly web interface where users can ask any question about Rome.

In [12]:
import gradio as gr


def ask_rome_guide(user_question):
    """Process a question through the RAG chain and return a formatted answer."""
    if not user_question or user_question.strip() == "":
        return "Please enter a question about Rome!"

    result = qa_chain.invoke({"question": user_question})

    answer = result.get("answer", "Sorry, I could not find an answer.").strip()
    sources = result.get("sources", "No sources found").strip()

    formatted = f"{answer}\n\n---\n**Sources:** {sources}"
    return formatted


demo = gr.Interface(
    fn=ask_rome_guide,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask me anything about Rome...",
        label="Your Question"
    ),
    outputs=gr.Markdown(label="Answer"),
    title="Rome Travel Guide AI",
    description="Ask questions about attractions, food, transport, neighborhoods, and practical tips for visiting Rome. Powered by RAG with LangChain and ChromaDB.",
    examples=[
        ["What are the must-see attractions in Rome?"],
        ["What are the four classic Roman pasta dishes?"],
        ["How do I get from Fiumicino airport to the city center?"],
        ["Is it safe to walk around Rome at night?"],
        ["What is the best time of year to visit Rome?"],
        ["Can I drink water from public fountains?"],
        ["What day trips can I take from Rome?"]
    ],
    flagging_mode="never"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
